# 🚀 Data Ingestion (TEAM VERSION)

Compatible con setup del equipo. No modifica tablas, solo añade datos.

In [ ]:
import os
from google.cloud import bigquery
from faker import Faker
import pandas as pd
import random
from datetime import timedelta

# VARIABLES DEL PROYECTO (compatibles con el equipo)
os.environ["GCP_PROJECT_ID"] = "project-468c9077-a9c5-4e11-b14"
os.environ["BQ_DATASET_ID"] = "tc_dataset"

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

assert PROJECT_ID is not None, "Falta PROJECT_ID"
assert DATASET_ID is not None, "Falta DATASET_ID"

client = bigquery.Client(project=PROJECT_ID, location="EU")

fake = Faker()

BASE_ID = 10000


## 👥 Clientes

In [ ]:
customers = []

for i in range(500):
    customers.append({
        "customer_id": BASE_ID + i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "country": fake.country_code(),
        "city": fake.city(),
        "adquisition_channel": random.choice(["organic","paid_ads","social","referral"]),
        "registration_date": fake.date_time_between(start_date='-2y', end_date='now')
    })

df_customers = pd.DataFrame(customers)


## 🛍️ Productos

In [ ]:
products = []

for i in range(70):
    price = round(random.uniform(10, 300), 2)

    products.append({
        "product_id": BASE_ID + i,
        "category_id": random.randint(1,10),
        "product_name": fake.word().capitalize(),
        "brand": fake.company(),
        "sale_price": price,
        "cost_price": round(price * random.uniform(0.4, 0.7), 2),
        "stock": random.randint(20,200),
        "is_active": True
    })

df_products = pd.DataFrame(products)


## 📦 Orders + Items

In [ ]:
orders = []
order_items = []

order_id = BASE_ID
order_item_id = BASE_ID

for i in range(2000):
    order_date = fake.date_time_between(start_date='-1y', end_date='now')

    orders.append({
        "order_id": order_id,
        "customer_id": random.choice(df_customers["customer_id"]),
        "order_date": order_date,
        "order_status": "delivered"
    })

    for _ in range(random.randint(2,3)):
        product = random.choice(products)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": random.randint(1,4),
            "unit_price": product["sale_price"]
        })

        order_item_id += 1

    order_id += 1

df_orders = pd.DataFrame(orders)
df_order_items = pd.DataFrame(order_items)


## 💳 Pagos

In [ ]:
payments = []

for o in orders:
    total = sum([
        oi["quantity"] * oi["unit_price"]
        for oi in order_items if oi["order_id"] == o["order_id"]
    ])

    payments.append({
        "payment_id": o["order_id"],
        "order_id": o["order_id"],
        "payment_method": random.choice(["credit_card","paypal","bank_transfer"]),
        "payment_status": "completed",
        "payment_date": o["order_date"],
        "amount": round(total,2)
    })

df_payments = pd.DataFrame(payments)


## 🚚 Shipments

In [ ]:
shipments = []

for o in orders:
    shipments.append({
        "shipments_id": o["order_id"],
        "order_id": o["order_id"],
        "shipping_status": "delivered",
        "shipping_country": fake.country_code(),
        "shipping_city": fake.city(),
        "delivery_date": o["order_date"] + timedelta(days=random.randint(1,5)),
        "street_address": fake.street_address(),
        "cp_code": fake.postcode(),
        "phone": fake.phone_number()
    })

df_shipments = pd.DataFrame(shipments)


## ⭐ Reviews

In [ ]:
reviews = []
review_id = BASE_ID

for oi in order_items:
    if random.random() < 0.35:
        reviews.append({
            "review_id": review_id,
            "customer_id": random.choice(df_customers["customer_id"]),
            "product_id": oi["product_id"],
            "rating": random.randint(1,5),
            "review_date": fake.date_time_between(start_date='-6m', end_date='now'),
            "comment": fake.sentence()
        })
        review_id += 1

df_reviews = pd.DataFrame(reviews)


## ☁️ Upload (APPEND)

In [ ]:
from google.cloud import bigquery

def upload(df, table_name):
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        )
    )

    job.result()
    print(f"{table_name} añadida correctamente")

upload(df_customers, "customers")
upload(df_products, "products")
upload(df_orders, "orders")
upload(df_order_items, "order_items")
upload(df_payments, "payments")
upload(df_shipments, "shipments")
upload(df_reviews, "reviews")
